# Notebook 02 — Mendelian Edge Cases: Linkage and Epistasis

**Module:** 06 — Genetics and Evolution  
**Notebook:** 02 of 12  
**Prerequisites:** Module 05 NB05 (Punnett squares, Mendel's laws)  
**Time estimate:** 75 minutes

> **Why this matters for GWAS:** Linkage disequilibrium (LD) is the reason GWAS uses
> tag SNPs rather than sequencing every variant. Understanding LD starts here.

---
## Step 1 — Motivation

Mendel's Law of Independent Assortment holds only for genes on *different chromosomes*.
Genes on the *same* chromosome travel together unless separated by recombination —
this is **linkage**. Meanwhile, **epistasis** means one gene's effect depends on
another's genotype. Both phenomena break the simple Mendelian ratios and must be
accounted for in any real genetic analysis.

---
## Step 3 — Biological Background

### Linkage and Recombination

Genes on the same chromosome tend to be inherited together (linked). During meiosis,
homologous chromosomes can exchange segments — **crossing over** — creating recombinant
chromosomes. The **recombination fraction** θ between two loci is the probability of
an odd number of crossovers between them:

- θ = 0.5 → independent assortment (Mendel's law holds)
- θ < 0.5 → linked (travel together more than expected)
- θ = 0 → completely linked (never separated)

**Genetic distance (Morgans):** 1 Morgan = 100 cM = 1 expected crossover per meiosis
between the two loci. The Haldane mapping function converts genetic distance d (in
Morgans) to recombination fraction: θ = (1 − e^{−2d}) / 2.

### Linkage Disequilibrium (LD)

LD is the non-random *association* of alleles at two loci in a population — even if
the loci are on the same chromosome, LD erodes over time due to recombination.

**D statistic:** D = f(AB) − f(A)·f(B) where f(AB) is the observed haplotype frequency.
D = 0 means alleles are independent; D ≠ 0 means they are associated.

**r² statistic:** r² = D² / (p_A(1-p_A)·p_B(1-p_B)). r² = 1 means perfect LD
(the two alleles are perfectly correlated across the population).

### Epistasis

When the phenotypic effect of alleles at one locus depends on the genotype at another
locus. Classic example: coat colour in Labrador retrievers (E and B loci interact).
In GWAS, epistatic interactions are rarely detected due to power issues — single-locus
GWAS misses interaction effects unless the study is specifically designed for them.

---
## Step 4 — Mathematical Explanation

**Haldane mapping function:**
$$\theta = \frac{1 - e^{-2d}}{2}$$

where d = genetic distance in Morgans.

**LD decay over generations:** Under random mating, LD decays each generation:
$$D_t = D_0 \cdot (1 - \theta)^t$$

**r² as a correlation:** r² between two SNPs tells you how much variation in one
explains variation in the other. In GWAS, if r² > 0.8 between a causal SNP and a
tag SNP, the tag SNP will 'capture' the signal.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — Haldane mapping function and genetic distance
def haldane_theta(d_morgans: float) -> float:
    """Convert genetic distance (Morgans) to recombination fraction."""
    return (1 - np.exp(-2 * d_morgans)) / 2

def haldane_distance(theta: float) -> float:
    """Convert recombination fraction to genetic distance (Morgans)."""
    if theta >= 0.5:
        return np.inf  # unlinked
    return -0.5 * np.log(1 - 2 * theta)

# Example: chromosome arm with genes at different distances
distances_cM = [0, 5, 10, 25, 50, 100, 200]
print(f"{'Distance (cM)':>15} {'Distance (M)':>13} {'Recomb. fraction (θ)':>22}")
print("-" * 52)
for d_cM in distances_cM:
    d_M = d_cM / 100
    theta = haldane_theta(d_M)
    print(f"{d_cM:>15} {d_M:>13.2f} {theta:>22.4f}")
print("\nNote: θ approaches 0.5 (free recombination) for large distances")

In [ ]:
# Cell 6.2 — LD statistics: D and r²
def ld_statistics(haplotype_counts: dict) -> dict:
    """
    Compute LD statistics D and r² from haplotype counts.

    haplotype_counts: dict with keys 'AB', 'Ab', 'aB', 'ab'
    """
    total = sum(haplotype_counts.values())
    f = {k: v / total for k, v in haplotype_counts.items()}

    p_A = f['AB'] + f['Ab']  # frequency of allele A at locus 1
    p_B = f['AB'] + f['aB']  # frequency of allele B at locus 2

    D = f['AB'] - p_A * p_B

    denom = p_A * (1 - p_A) * p_B * (1 - p_B)
    r2 = D**2 / denom if denom > 0 else 0.0

    D_prime = D / (D_max(D, p_A, p_B))

    return {'D': D, 'r2': r2, 'D_prime': D_prime,
            'p_A': p_A, 'p_B': p_B, 'f_AB': f['AB']}

def D_max(D, p_A, p_B):
    if D > 0:
        return min(p_A * (1 - p_B), (1 - p_A) * p_B)
    else:
        return max(-p_A * p_B, -(1 - p_A) * (1 - p_B))

# Example 1: Perfect LD (r2 = 1)
perfect_ld = {'AB': 400, 'Ab': 0, 'aB': 0, 'ab': 600}
r1 = ld_statistics(perfect_ld)
print(f"Perfect LD:      D={r1['D']:.4f}, r²={r1['r2']:.4f}, D'={r1['D_prime']:.4f}")

# Example 2: Linkage equilibrium
equil = {'AB': 240, 'Ab': 160, 'aB': 360, 'ab': 240}
r2 = ld_statistics(equil)
print(f"Equilibrium:     D={r2['D']:.4f}, r²={r2['r2']:.4f}, D'={r2['D_prime']:.4f}")

# Example 3: Partial LD
partial = {'AB': 350, 'Ab': 50, 'aB': 50, 'ab': 550}
r3 = ld_statistics(partial)
print(f"Partial LD:      D={r3['D']:.4f}, r²={r3['r2']:.4f}, D'={r3['D_prime']:.4f}")

In [ ]:
# Cell 6.3 — LD decay over generations
def ld_decay_simulation(D0: float, theta: float, n_gen: int) -> np.ndarray:
    """Simulate deterministic LD decay: D_t = D0 * (1-theta)^t"""
    t = np.arange(n_gen + 1)
    return D0 * (1 - theta)**t

n_gen = 200
theta_values = [0.001, 0.01, 0.05, 0.5]
print("LD half-lives (generations to reach D0/2):")
for theta in theta_values:
    half_life = np.log(0.5) / np.log(1 - theta)
    print(f"  θ={theta:.3f}: ~{half_life:.1f} generations")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Haldane function + LD decay
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel 1: Haldane mapping function
ax = axes[0]
d_range = np.linspace(0, 2, 300)
theta_range = np.array([haldane_theta(d) for d in d_range])
ax.plot(d_range * 100, theta_range, color='steelblue', lw=2, label='Haldane')
ax.axhline(0.5, color='gray', ls='--', lw=1, label='θ=0.5 (free recomb.)')
ax.set_xlabel('Genetic distance (cM)')
ax.set_ylabel('Recombination fraction θ')
ax.set_title('Haldane mapping function:\nθ saturates at 0.5 for distant loci')
ax.legend()

# Panel 2: LD decay for different recombination rates
ax = axes[1]
colors_ld = ['tomato', 'orange', 'steelblue', 'seagreen']
t_axis = np.arange(201)
for theta, color in zip(theta_values, colors_ld):
    d_t = ld_decay_simulation(D0=0.25, theta=theta, n_gen=200)
    ax.plot(t_axis, d_t, color=color, lw=2, label=f'θ={theta}')
ax.set_xlabel('Generations')
ax.set_ylabel('LD coefficient D')
ax.set_title('LD decay over generations:\nhigher recombination → faster erosion')
ax.legend()

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `haldane_theta(d_morgans)` and `ld_statistics(haplotype_counts)` from scratch.
   Verify: for θ=0.1, how many generations until D is less than 5% of D0?
2. In a GWAS, tag SNPs capture causal variants using LD (r² > 0.8). Why would using
   only HapMap SNPs as tags miss rare variants (allele frequency < 5%)?
3. Epistasis: two loci (A and B) independently have no effect on disease risk, but
   the combination aaBB causes disease. Draw the Punnett square for a dihybrid cross
   (AaBb × AaBb) and calculate the expected fraction of affected offspring.
4. What is the Felsenstein zone and why is it relevant to linkage analysis?

---
## Quiz — Active Recall

1. What does a recombination fraction of θ = 0.1 mean?
2. What is LD? Why does it decay over generations?
3. What does r² = 1 between two SNPs tell you?
4. What is epistasis? Give a biological example.
5. Why do GWAS typically use r² > 0.8 as their LD threshold for tag SNPs?

---
## Reflection

**Date completed:** ____________________

> *[A GWAS paper reports that a significant SNP is in LD (r²=0.92) with a coding
> variant in gene X. Does this mean the coding variant is causal? What else would
> you need to know?]*

---
*Next: `03_molecular_basis_of_mutation.ipynb`*